In [12]:
import sys
import os 
import pandas as pd
import numpy as np
from omegaconf import OmegaConf
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise_distances
import random
import itertools
from scipy.stats import qmc
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from src.analysis.sensitivity import SensitivityAnalyzer

In [13]:
sys.path.append(os.path.abspath("../external/AlphaPEM"))
from model.AlphaPEM import AlphaPEM
from src.sampling.sampler import get_polarisation_curve_samples, build_fixed_parameters

In [15]:
parameter_ranges = OmegaConf.load('../param_config.yaml')
sampler = qmc.LatinHypercube(d=len(parameter_ranges.keys()), seed=42)
parameters = list(parameter_ranges.keys())
sample = sampler.random(10000)
sample = pd.DataFrame(sample, columns=parameters)

dependent_parameter_names = ['Pc_des']
dependent_parameters = [{'parameter_name': 'Pc_des', 'function': lambda Pa_des : Pa_des - 20000, 'dependent_param': 'Pa_des'}]

In [16]:
SA = SensitivityAnalyzer(parameter_ranges, dependent_parameter_names=None, method='sobol', seed=42, N=1024 , calculate_second_order=True)
SA.samples_df = sample
SA.rescale_samples()
SA.define_id()
samples = SA.apply_dependent_parameters(dependent_parameters)

In [17]:
pd.DataFrame(samples).to_pickle("../data/raw/LHS_sampling_10000.pkl")
#all_dejvis = pd.read_pickle("../data/raw/LHS_sampling_10000.pkl")

In [6]:
from src.sampling.sampler import get_polarisation_curve_samples, build_fixed_parameters

In [7]:
save_filepath = "../sampling_test/prueba_w_id.pkl"
results = get_polarisation_curve_samples(sampled_parameters=samples.iloc[:2,:].to_dict(orient='records'), fixed_parameters = build_fixed_parameters(), save_path=save_filepath, save_every=10)#


📁 Final save complete: ../sampling_test/prueba_w_id.pkl with 2 samples.


In [ ]:
import concurrent.futures
import math
import uuid
import os

def run_single_sample(sample, fixed_params, save_path):
    unique_id = uuid.uuid4().hex
    unique_file = save_path+f"sample_result_{unique_id}.pkl"

    result = get_polarisation_curve_samples(
        sampled_parameters=[sample],
        fixed_parameters=fixed_params,
        save_path=unique_file,
        save_every=1
    )
    return result

def run_in_parallel(samples, fixed_params, save_path, max_workers=8):
    results = []
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(run_single_sample, sample, fixed_params, save_path)
            for sample in samples
        ]
        for i, future in enumerate(concurrent.futures.as_completed(futures)):
            res = future.result()
            results.append(res)
            print(f"Completed {i+1}/{len(samples)}")
    return results

# Usage:
samples = samples_add.to_dict(orient='records')
fixed_params = build_fixed_parameters()
save_filepath ="../data/raw/extra_1024_part3/"

all_results = run_in_parallel(samples, fixed_params, save_filepath, max_workers=8)

In [ ]:
import hashlib
def generate_row_id(row):
    # Convert all values to string, concatenate, and hash
    row_string = '|'.join([str(x) for x in row])
    return hashlib.sha256(row_string.encode()).hexdigest()
df['SHA256'] = df.apply(generate_row_id, axis=1)